In [ ]:
from verl.utils.megatron_utils import (
    load_megatron_model_to_gpu,
    load_megatron_optimizer,
    offload_megatron_model_to_cpu,
    offload_megatron_optimizer,
)

from verl.utils.checkpoint.megatron_checkpoint_manager import MegatronCheckpointManager

In [ ]:
    @register(dispatch_mode=Dispatch.ONE_TO_ALL)
    def init_model(self):
            self.checkpoint_mananager = MegatronCheckpointManager(
            config=self.config,
            checkpoint_config=self.config.checkpoint,
            model_config=self.critic_model_config,
            transformer_config=self.tf_config,
            role="critic",
            model=self.critic_module,
            arch=self.architectures[0],
            hf_config=self.hf_config,
            param_dtype=self.param_dtype,
            share_embeddings_and_output_weights=False,
            processing_class=self.processor if self.processor is not None else self.tokenizer,
            optimizer=self.critic_optimizer,
            optimizer_scheduler=self.critic_optimizer_scheduler,
            use_distributed_optimizer=self.config.megatron.use_distributed_optimizer,
            use_checkpoint_opt_param_scheduler=self.config.optim.use_checkpoint_opt_param_scheduler,
            bridge=self.bridge,
            use_dist_checkpointing=self.config.megatron.use_dist_checkpointing,
        )

# load_checkpoint()

- 这段代码是一个典型的支持“训推共卡”（训练与推理共享显存）以及远程存储（HDFS）的模型权重加载函数。
- 它通常出现在大模型强化学习（如 PPO、GRPO）的 Actor 模型更新阶段。
- 核心逻辑是：在加载权重的短暂窗口期内，将模型和优化器从 CPU 移回 GPU，加载完成后迅速卸载回 CPU，从而把宝贵的 GPU 显存资源让给其他计算任务（比如 Critic 模型或 Reward 模型）。
- 结合你之前了解的 vLLM 和 SGLang 的“睡眠/唤醒”机制，这段代码其实是同一个设计理念在 Megatron 框架下的具体落地。以下是逐行详细解析：

## 🎯 核心参数解析
- checkpoint_path：本地检查点（权重文件）的路径。
- hdfs_path：<font color='red'>远程 HDFS 存储的路径。</font>在实际的大规模训练中，权重通常归档在 HDFS，加载时会自动拉取到本地。
- del_local_after_load=True：一个非常实用的工程化参数。表示<font color='red'>权重从 HDFS 拉取到本地并加载完毕后，自动删除本地副本，防止本地磁盘空间被大量的历史权重文件占满。</font>

- <font color='red'>“CPU 卸载 -> 临时搬回 GPU 加载-> 加载完成 -> 再次卸载回 CPU”</font>

In [ ]:
    @register(dispatch_mode=Dispatch.ONE_TO_ALL)
    def load_checkpoint(self, checkpoint_path, hdfs_path=None, del_local_after_load=True):
        '''
        唤醒模型参数（如果处于卸载状态）:
        
        背景：在训推共卡场景下，Actor 模型的权重平时是卸载（Offload）在 CPU 内存中的（即 self._is_offload_param 为 True），
             以腾出 GPU 显存。
        
        作用：在加载新权重之前，必须先把模型“搬回”GPU。
            - 因为 Megatron 底层的 load_checkpoint 接口通常要求模型必须位于 GPU 设备上才能进行权重的映射和覆盖。
        '''
        if self._is_offload_param:
            load_megatron_model_to_gpu(self.actor_module)
            
        '''
        执行核心加载动作:
        
        作用：这是真正的加载环节。checkpoint_manager 会处理复杂的逻辑，例如：
            - 检查 hdfs_path 是否存在，如果存在且本地没有，则从 HDFS 下载。
            - 调用 Megatron 框架原生的加载接口，将磁盘上的权重文件读取并覆盖到当前 GPU 上的 actor_module 中。
            - 加载完成后，根据 del_local_after_load 的指令清理本地临时文件。
        '''
        self.checkpoint_mananager.load_checkpoint(
            local_path=checkpoint_path, hdfs_path=hdfs_path, del_local_after_load=del_local_after_load
        )
        '''
        再次卸载模型参数（释放显存）:
        
        作用：权重加载完毕后，立刻将模型权重从 GPU 搬回 CPU 内存。
        
        目的：这一步对应了你之前了解的 release_memory_occupation 或 sleep 的逻辑。加载完最新的权重后，
            Actor 模型通常不需要立刻在 GPU 上进行推理，因此迅速释放显存，为后续的梯度计算或其他模型（如 Critic）让路。
        '''
        if self._is_offload_param:
            offload_megatron_model_to_cpu(self.actor_module)
        if self._is_offload_optimizer:
            offload_megatron_optimizer(self.actor_optimizer)

## MegatronCheckpointManager
- verl/utils/checkpoint/gegatron_checkpoint_manager.py

### 类名： MegatronCheckpointManager（继承自基础检查点管理器 BaseCheckpointManager）
- 核心描述：
    - 这是专为 Megatron-LM 分布式训练环境设计的检查点管理器。
    - 它负责处理模型检查点的保存与加载，涵盖了模型状态、优化器状态、学习率调度器以及随机数生成器（RNG）状态等各个方面，并确保与 HuggingFace 格式兼容。

- 核心功能 (Key features)：
    - <b>分布式检查点存取：</b> <font color='red'>利用 Megatron 的 dist_checkpointing 技术进行高效的分布式保存和加载。</font>
    - <b>并行配置支持：</b> <font color='red'>完美支持张量并行（TP）、流水线并行（PP）和数据并行（DP）等多种复杂的并行配置。</font>
    - <b>多阶段模型状态处理</b>： 能够自动处理跨越多个流水线阶段的模型状态字典（state dictionaries）。
    - <b>HuggingFace 集成</b>： <font color='red'>深度集成了 HuggingFace 的模型配置和分词器（Tokenizer）。</font>
    - <b>随机数状态管理</b>： 管理 RNG 状态，确保训练过程的可复现性。
    - <b>同步与异步支持</b>： 同时支持同步和异步的检查点操作（<font color='red'>异步保存可以极大减少训练阻塞时间</font>）。

- 自动处理事项 (The manager automatically handles)：
    - 目录结构创建： 根据<b>全局步数（global steps</b>）和<b>进程排名（process ranks）</b>自动创建规范的存档目录。
    - 格式转换保存： <font color='red'>自动以 HuggingFace 格式保存模型配置和分词器。</font>
    - 状态持久化： 负责优化器和学习率调度器状态的持久化存储。
    - CUDA RNG 管理： 管理 CUDA 随机数状态，保证确定性训练（<font color='red'>即同样的种子能跑出同样的结果</font>）。
    - 清理与保留策略： 自动执行检查点的清理工作（比如只保留最近的 N 个存档），防止磁盘爆满。

- 参数 (Args)：
    - model：需要进行存档的 Megatron 模型实例。
    - optimizer：优化器实例（可选）。
    - lr_scheduler：学习率调度器实例（可选）。

- 属性 (Attributes)：
    - model：被存档的 Megatron 模型的引用。
    - optimizer：优化器的引用（如果提供了的话）。
    - lr_scheduler：学习率调度器的引用（如果提供了的话）。
    - rank：当前分布式设置中的进程排名。

### 💡 为什么需要这个管理器？

<font color='red'>在训练拥有数百亿甚至上万亿参数的大模型时，直接调用 PyTorch 原生的 torch.save 是非常低效甚至不可行的。</font>

MegatronCheckpointManager 解决了以下几个痛点：
1. <b>显存与效率问题</b>： 
    - 传统的保存方式需要把所有 GPU 上的数据汇总到一张卡上再写入磁盘，速度极慢且容易撑爆显存。
    - <font color='red'>这个管理器利用了 Megatron 的分布式检查点技术，让每张卡只保存自己负责的那部分数据（分片），写入速度极快。</font>
2. <b>弹性恢复（Resharding）</b>： 它支持“弹性”。
    - <font color='red'>比如你之前用 32 张卡（TP=4）保存了检查点，现在想用 64 张卡（TP=8）来恢复训练，这个管理器能自动进行张量的重新切分和映射，无需从头开始训练。</font>
3. <b>容错与复现</b>： 大模型训练动辄数周，硬件故障难以避免。<font color='red'>它能完整保存优化器状态和随机数种子，确保任务中断后能从断点处“无缝衔接”继续训练，且结果完全一致。</font>


In [ ]:
class MegatronCheckpointManager(BaseCheckpointManager):
    
        def __init__(
        self,
        config,
        checkpoint_config,
        model_config,
        transformer_config,
        role,
        model: torch.nn.ModuleList,
        arch: str,
        hf_config,
        param_dtype: torch.dtype,
        share_embeddings_and_output_weights: bool,
        processing_class,
        optimizer,
        optimizer_scheduler,
        use_distributed_optimizer: bool,
        use_checkpoint_opt_param_scheduler: bool = False,
        use_dist_checkpointing: bool = True,
        bridge=None,
        **kwargs,
    ):
        '''
        这部分调用了父类（BaseCheckpointManager）的初始化方法，将模型（model）、优化器（optimizer）
        、学习率调度器（optimizer_scheduler）等最核心的训练组件进行注册和托管。这确保了管理器能够访问并控制这些组件的状态。
        '''
        super().__init__(
            model,
            optimizer=optimizer,
            lr_scheduler=optimizer_scheduler,
            processing_class=processing_class,
            checkpoint_config=checkpoint_config,
        )
        self.arch = arch  # 模型的架构名称（如 llama, qwen 等），用于后续匹配对应的权重加载器。
        self.config = config
        self.transformer_config = transformer_config
        
        '''
        如果角色是 reward 或 critic，则将其标记为 is_value_model（价值模型），
        这在后续处理模型头（Head）或权重加载逻辑时会有不同的处理分支
        '''
        self.role = role
        self.is_value_model = False
        if self.role in ["reward", "critic"]:
            self.is_value_model = True
            
        self.model_config = model_config
        self.hf_config = hf_config # 分别存储 HuggingFace 格式和 Megatron 内部的模型配置
        self.param_dtype = param_dtype
        self.share_embeddings_and_output_weights = share_embeddings_and_output_weights # 一个布尔值，标记是否共享输入嵌入层（Embedding）和输出层（LM Head）的权重（这在 LLaMA 等模型中很常见）
        self.model_path = self.config.model.path
        self.use_distributed_optimizer = use_distributed_optimizer  # 标记是否使用了分布式优化器（如 Megatron 的 DistributedOptimizer），这会影响优化器状态的保存方式。
        self.use_checkpoint_opt_param_scheduler = use_checkpoint_opt_param_scheduler
        self.bridge = bridge
        self.rank = torch.distributed.get_rank() # 获取当前进程在分布式环境中的全局编号，用于区分不同 GPU 的保存路径。
        
        '''
        use_dist_checkpointing（是否使用分布式检查点）：
            - 如果显式开启了 use_dist_checkpointing；
            - 或者没有使用桥接器（not self.bridge，即纯 Megatron 模式）； 系统完全依赖 Megatron 自身的原生机制来处理模型
            - 或者当前是价值模型（self.is_value_model）；
            - 只要满足以上任一条件，就会启用分布式检查点（Distributed Checkpointing）。
              这种方式允许每张 GPU 只保存自己负责的那部分权重分片，极大提升了大规模模型训练的 IO 效率和弹性恢复能力。
        '''
        self.use_dist_checkpointing = use_dist_checkpointing or not self.bridge or self.is_value_model
        '''
        use_hf_checkpoint（是否使用 HuggingFace 格式检查点）：
            - 如果不满足上述分布式条件，则回退到使用标准的 HuggingFace 格式进行权重的保存与加载。这通常用于模型初始化阶段或特定的微调场景。

        '''
        self.use_hf_checkpoint = not self.use_dist_checkpointing

        '''
        权重保存器实例化:
        
        最后，根据模型的架构（arch），通过工厂函数 get_weight_saver 动态获取对应的权重保存器。
        不同的模型架构（如 Transformer 的不同变体）在序列化权重时可能有细微差别，这一步确保了保存逻辑的精准匹配。

        '''
        self.weight_saver = get_weight_saver(self.arch)


### self.checkpoint_mananager.load_checkpoint

- 这段 load_checkpoint 代码是整个 MegatronCheckpointManager 最核心的“读档”逻辑。
- 它的主要职责是：根据当前的运行模式（纯 Megatron 分布式模式还是 HuggingFace 桥接模式），将保存在磁盘上的模型权重、优化器状态以及随机数种子等数据，精准地恢复到内存中，确保训练能够无缝衔接。

- 这段代码展现了大模型训练中极高的工程严谨性：
    - 它不仅通过“预生成状态字典”解决了分布式环境下的权重精准映射问题，
    - 还通过双分支逻辑兼容了原生高性能训练与 HuggingFace 生态，
    - 同时兼顾了训练状态（优化器、RNG）的完整恢复与磁盘空间的自动化管理。

In [ ]:
from verl.utils.megatron.dist_checkpointing import load_dist_checkpointing, save_dist_checkpointing

def load_checkpoint(self, local_path: str, hdfs_path: str = None, del_local_after_load=False):
        
        '''
        路径校验与状态字典预生成:
        
        - 路径检查：首先确保传入的本地检查点路径真实存在，防止后续读取报错。
        - 获取分布式路径：通过 get_dist_checkpoint_path 解析出分布式检查点的具体存储目录。
        - 生成空壳状态字典（sharded_state_dict）：这里调用 generate_state_dict() 并不是为了保存，
            而是为了构建一个与当前模型结构完全匹配的“接收容器”。在 Megatron 的分布式加载机制中，
            框架需要预先知道每张 GPU 上应该接收哪些权重分片（Shard），因此必须先根据当前的 TP（张量并行）、PP（流水线并行）配置生成这个映射结构。
        
        '''
        if local_path is not None:
            assert os.path.exists(local_path), f"Checkpoint path {local_path} does not exist."

        dist_checkpoint_path = get_dist_checkpoint_path(local_path)

        
        # Get State Dict for loading
        sharded_state_dict = self.generate_state_dict()
        log_with_rank(f"Generated state dict for saving: {sharded_state_dict.keys()}", rank=self.rank, logger=logger)
        for vpp_rank, model in enumerate(self.model):
            if len(self.model) > 1:
                model_i_keys = sharded_state_dict[f"model{vpp_rank}"].keys()
                log_with_rank(f"Generated state dict for saving: {model_i_keys}", rank=self.rank, logger=logger)
            else:
                log_with_rank(
                    f"Generated state dict for saving: {sharded_state_dict['model'].keys()}",
                    rank=self.rank,
                    logger=logger,
                )

        '''
        调用底层分布式加载接口:
        
        这里调用了 Megatron 底层的 load_dist_checkpointing 函数。它会根据上一步生成的“接收容器”，
        从磁盘的 dist_checkpoint_path 中并行读取对应的权重分片，
        并将其填充到 state_dict 中。这一步充分利用了分布式 IO 的优势，避免了将所有权重汇总到单张卡导致的显存溢出。
        
        '''
        # Load Dist Checkpointing
        state_dict = load_dist_checkpointing(
            sharded_state_dict=sharded_state_dict,
            ckpt_dir=dist_checkpoint_path,
        )

        '''
        模型权重的双模式加载（核心分支）:
        
        代码根据初始化时确定的策略（use_dist_checkpointing 还是 use_hf_checkpoint），进入了两个截然不同的加载分支：
        '''
        if self.should_load_model and self.use_dist_checkpointing:
            '''
            分支 A：纯 Megatron 分布式加载（use_dist_checkpointing）
            '''
            assert "model" in state_dict or any(
                f"model{vpp_rank}" in state_dict for vpp_rank in range(len(self.model))
            ), f"Model state dict not found in {state_dict.keys()}. Please check the checkpoint file {local_path}."
            
            '''
            ... 校验 model 或 model0, model1 等键是否存在 ...
            '''
            for vpp_rank, model in enumerate(self.model):
                '''
                处理流水线并行（VPP）的多阶段模型
                '''
                if len(self.model) == 1:
                    model_state_dict = state_dict["model"]
                else:
                    assert f"model{vpp_rank}" in state_dict, f"model{vpp_rank} not found in state_dict"
                    model_state_dict = state_dict[f"model{vpp_rank}"]
                
                '''
                设置当前流水线阶段的并行排名，确保权重映射到正确的 GPU 分片上
                
                这是最标准的 Megatron 恢复流程。它会遍历模型的所有流水线阶段（VPP rank），
                 利用 mpu.set_virtual_pipeline_model_parallel_rank 确保每张卡只加载属于自己那部分的权重分片，
                 完美适配 TP/PP 并行环境。
                '''
                mpu.set_virtual_pipeline_model_parallel_rank(vpp_rank)
                self.model[vpp_rank].load_state_dict(model_state_dict)
            log_with_rank(f"Loaded sharded model checkpoint from {local_path}", rank=self.rank, logger=logger)
        elif self.should_load_model and self.use_hf_checkpoint:
            '''
            分支 B：HuggingFace 桥接加载（use_hf_checkpoint）
            
            如果处于桥接模式，代码会直接调用 self.bridge.load_weights。
             这意味着它加载的是标准的 HuggingFace 格式权重（如 pytorch_model.bin），
             并通过桥接器自动完成从 HF 格式到 Megatron 内部结构的转换和映射。
            '''
            hf_model_path = get_hf_model_checkpoint_path(local_path)
            self.bridge.load_weights(self.model, hf_model_path)
            log_with_rank(f"Loaded HF model checkpoint from {hf_model_path} with bridge", rank=self.rank, logger=logger)

        
        '''
        优化器、调度器与随机数状态的恢复:
        
        为了保证训练的完全可复现性（即从断点恢复后，后续产生的随机数序列与未中断时完全一致），除了模型权重，
        还必须完整恢复优化器的动量/方差状态、学习率调度器的当前步数，以及 CUDA 的随机数生成器状态（RNG State）
        '''
        if self.should_load_optimizer:
            assert "optimizer" in state_dict, (
                f"Optimizer state dict not found in {state_dict.keys()}. Please check the checkpoint file {local_path}."
            )
            optimizer_state_dict = state_dict["optimizer"]
            '''
            ... 校验并加载 optimizer ...
            '''
            self.optimizer.load_state_dict(optimizer_state_dict)
            log_with_rank(f"Loaded optimizer checkpoint from {local_path}", rank=self.rank, logger=logger)
            '''
            如果配置了保存优化器参数调度器，则继续加载学习率调度器
            '''
            if self.use_checkpoint_opt_param_scheduler:
                assert "lr_scheduler" in state_dict, (
                    f"LR scheduler state dict not found in {state_dict.keys()}. Please check the checkpoint file "
                    f"{local_path}."
                )
                lr_scheduler_state_dict = state_dict["lr_scheduler"]
                if self.lr_scheduler is not None:
                    self.lr_scheduler.load_state_dict(lr_scheduler_state_dict)
                    log_with_rank(f"Loaded LR scheduler checkpoint from {local_path}", rank=self.rank, logger=logger)

        if self.should_load_extra:
            assert "rng_state" in state_dict, (
                f"RNG state dict not found in {state_dict.keys()}. Please check the checkpoint file {local_path}."
            )
            rng_state = state_dict["rng_state"]
            '''
            ... 校验并加载 rng_state（随机数种子状态） ...
            '''
            self.load_rng_states(rng_state)
            log_with_rank(f"Loaded RNG states from {local_path}", rank=self.rank, logger=logger)

        '''
        临时文件清理:
        
        最后，如果开启了 del_local_after_load（通常用于从 HDFS 等远程存储拉取临时检查点的场景），
        在确认加载成功后，会自动删除本地的临时检查点文件，防止磁盘空间被大量历史存档占满。
        '''
        if del_local_after_load:
            try:
                os.remove(local_path) if is_non_local(local_path) else None
            except Exception as e:
                log_with_rank(
                    f"remove local resume ckpt file after loading failed, exception {e} will be ignored",
                    rank=self.rank,
                    logger=logger,
                )


#### load_dist_checkpointing(sharded_state_dict, ckpt_dir):


In [ ]:
from megatron.core import dist_checkpointing, mpu
from megatron.core.dist_checkpointing.serialization import (
    get_default_load_sharded_strategy,
    get_default_save_sharded_strategy,
)
from megatron.core.dist_checkpointing.strategies.fully_parallel import (
    FullyParallelLoadStrategyWrapper,
    FullyParallelSaveStrategyWrapper,
)

In [ ]:
def load_dist_checkpointing(sharded_state_dict, ckpt_dir):
    # Get checkpointing strategies
    load_strategy = get_default_load_sharded_strategy(ckpt_dir)
    load_strategy = FullyParallelLoadStrategyWrapper(
        load_strategy, mpu.get_data_parallel_group(with_context_parallel=True)
    )

    # Load model sharded state dicts
    state_dict = dist_checkpointing.load(sharded_state_dict, ckpt_dir, sharded_strategy=load_strategy)

    return state_dict

####    load_strategy = get_default_load_sharded_strategy(ckpt_dir)
- megatron.core.dist_checkpointing.serialization.get_default_load_sharded_strategy

- <font color='red'>是一个用于获取默认分片加载策略的辅助函数。</font>
- <font color='red'>它的核心作用是：在 Megatron 框架执行检查点加载（load）操作时，自动返回当前系统配置下最合适的加载策略对象。这个策略对象决定了 Megatron 底层如何从磁盘中读取、解析并重组被切分的模型权重。</font>
    
我们可以从以下几个维度来深入理解它：
##### ⚙️ 核心作用：加载策略的“调度员”
- 在 Megatron 的分布式检查点机制中，“策略（Strategy）”是一个核心抽象。它封装了具体的 IO 和数据处理逻辑。

- get_default_load_sharded_strategy 就像一个调度员，当加载流程启动时，它会告诉系统：“<font color='red'>按照目前的环境和配置，我们应该使用哪一套规则来读取检查点。”</font>

##### 🎯 返回的策略对象（以 PyTorch 原生格式为例）
在当前的 Megatron-Core 版本中，<font color='red'>最常用的默认加载策略通常是基于 PyTorch 分布式检查点格式的</font>。
- 调用该函数通常会返回一个 TorchDistLoadShardedStrategy 类的实例。

这个策略对象内部包含了具体的加载规划器（Planner），例如 MCoreLoadPlanner。它的主要职责包括：
- 读取元数据：解析检查点目录下的 .metadata 文件，<font color='red'>了解模型权重是如何被切分的。</font>
- 全局形状验证：MCoreLoadPlanner 会在默认规划器的基础上，增加对张量全局形状的验证，确保加载的权重分片与当前模型的架构配置完全匹配。
- 分片映射：<font color='red'>计算当前 GPU（Rank）应该从磁盘的哪些文件中读取哪一部分数据。</font>

##### 🔗 在加载流程中的位置
结合你之前看过的 load_checkpoint 代码，这个函数通常在底层的 load_dist_checkpointing 函数内部被调用。其工作流程如下：
- 上层调用 load_dist_checkpointing(sharded_state_dict, ckpt_dir)。
- 底层框架调用 get_default_load_sharded_strategy() <font color='red'>获取具体的加载策略。</font>
- 框架使用该策略对象，结合传入的 sharded_state_dict（预生成的接收容器），去 ckpt_dir 中并行读取并填充权重数据。

##### 💡 为什么需要“获取默认”策略？
你可能会问，为什么不直接把策略写死，而要设计一个 get_default_... 函数？这体现了框架的高度可扩展性：

- 兼容多种后端：<font color='red'>Megatron 不仅支持 PyTorch 原生的分布式格式，还可能支持其他存储后端或自定义的分片格式。通过这个函数，框架可以根据配置（例如是否启用了某些特定的异步 IO 库）动态切换底层的加载实现。</font>

- 用户自定义：<font color='red'>高级用户或开发者可以编写自己的加载策略（继承自基础策略类），并通过注册机制替换掉这个“默认”策略，从而实现特定的优化（比如从特殊的对象存储中极速加载）。</font>

总结来说，get_default_load_sharded_strategy <font color='red'>是 Megatron 分布式检查点加载机制中的“指南针”</font>。它对外隐藏了底层复杂的 IO 实现细节，对内确保了加载流程能够根据当前环境，精准地调用正确的逻辑来恢复模型权重。

#### FullyParallelLoadStrategyWrapper
- megatron.core.dist_checkpointing.strategies.fully_parallel.FullyParallelLoadStrategyWrapper

FullyParallelLoadStrategyWrapper 是 Megatron-Core 分布式检查点机制中最核心、最高效的 <font color='red'>加载策略包装器。</font>
- 它的核心作用是：在加载模型检查点时， <font color='red'>彻底打破传统“先汇总再分发”的瓶颈</font>，实现完全并行的权重加载。这意味着，在拥有成百上千张 GPU 的集群中， <font color='red'>每张 GPU 都可以独立、并行地从磁盘中读取自己负责的那一小部分权重分片</font>，从而极大地提升了大规模模型的恢复速度。

我们可以从以下几个维度来深入拆解它：
##### 🎯 核心痛点：为什么需要“完全并行加载”？
在传统的模型加载方式中，通常需要将所有 GPU 上的权重分片先汇总（Gather）到主进程（Rank 0），拼成一个完整的模型文件后再分发给其他 GPU。
- 痛点 1：单点瓶颈。Rank 0 的显存和磁盘 IO 压力巨大，极易导致 OOM（显存溢出）或加载超时。
- 痛点 2：扩展性差。 <font color='red'>随着 GPU 数量增加，汇总和分发的通信开销呈指数级上升，导致加载时间越来越长。</font>

FullyParallelLoadStrategyWrapper 正是为了解决这些问题而生。它利用了 Megatron 的 3D 并行（张量并行 TP、流水线并行 PP、数据并行 DP）拓扑结构， <font color='red'>让每张卡“各取所需”，只加载自己那部分数据。</font>

##### ⚙️ 工作原理：它是如何运作的？
FullyParallelLoadStrategyWrapper 本质上是一个包装器（Wrapper），它内部包裹了一个底层的加载策略（例如 TorchDistLoadShardedStrategy）。它的工作流程如下：
1. 拦截加载请求： <font color='red'>当上层的 load_dist_checkpointing 发起加载指令时，这个 Wrapper 会首先拦截。</font>
2. 结合并行拓扑：它会获取当前进程组（Process Group）的并行配置信息（比如当前是哪个 TP 组、哪个 PP 阶段）。
3. 智能分片映射： <font color='red'>结合底层的元数据（Metadata）和当前的并行拓扑，它会精准计算出：“在当前这张 GPU 上，我需要加载模型权重的哪一块切片”。</font>
4. 并行 IO 执行： <font color='red'>指挥内部的底层策略，让所有 GPU 同时向存储系统发起读取请求。每张卡只读取自己那几 MB 或几十 MB 的数据，互不干扰。</font>

##### 🔗 在加载流程中的位置
结合你之前了解的 get_default_load_sharded_strategy，在实际的 Megatron-Core 运行中，get_default_load_sharded_strategy 返回的往往就是被 FullyParallelLoadStrategyWrapper 包裹后的策略对象。

其调用链路大致如下：
load_checkpoint (用户层)
-> load_dist_checkpointing (中间层)
-> get_default_load_sharded_strategy (获取策略)
-> FullyParallelLoadStrategyWrapper (核心并行加载逻辑)
-> TorchDistLoadShardedStrategy (底层具体 IO 实现)

##### 💡 它的核心价值
- 极致的加载速度：在千卡级别的集群上，加载一个千亿参数的模型，使用完全并行加载策略可能只需要几分钟，而传统方式可能需要几十分钟甚至更久。
- 内存安全：由于不再需要将完整模型汇聚到单张卡，彻底消除了加载阶段的单卡 OOM 风险。
- 弹性恢复（Resharding）的基础： <font color='red'>它支持“异构加载”。例如，你可以用 64 张卡（TP=4）保存检查点，然后用 128 张卡（TP=8）来加载并继续训练。Wrapper 会自动处理不同并行度下的权重切片映射，无需人工干预。</font>

 <font color='red'>总结来说，FullyParallelLoadStrategyWrapper 是 Megatron 能够支撑超大规模（如 6000+ GPU）训练和快速弹性恢复的幕后功臣。</font>
 - 它通过将“集中式加载”转变为“分布式并行加载”，完美解决了大模型检查点恢复时的 IO 和显存瓶颈。

#### dist_checkpointing.load
- <font color='red'>from megatron.core import dist_checkpointing</font>

- load 方法是 Megatron-Core 分布式检查点机制对外暴露的最核心、最高层的加载接口。
- 简单来说，你在之前的代码中看到的底层 load_dist_checkpointing 或框架原生的 load_checkpoint，最终都会调用到这个 load 方法。它的核心职责是：在分布式训练环境中，将保存在磁盘上的模型、优化器等状态，精准、高效且安全地恢复到当前集群的各个 GPU 进程中。

##### 🎯 核心参数与功能定位
dist_checkpointing.load 方法的签名通常如下：
- sharded_state_dict（分片状态字典）：这是加载的核心。它不是一个空的字典，而是一个已经包含了当前模型在每张 GPU 上应有的权重分片形状、数据类型等元数据的“蓝图”或“接收容器”。load 方法会根据这个蓝图，去磁盘上寻找并填充对应的数据。
- checkpoint_dir：指定了分布式检查点保存的根目录。

In [ ]:
dist_checkpointing.load(
    sharded_state_dict,  # 预生成的“接收容器”
    checkpoint_dir,      # 检查点所在的目录路径
    validate_access_integrity=True, # 是否验证访问完整性
    ...
)

#### ⚙️ 内部工作流程（黑盒揭秘）
当调用 dist_checkpointing.load 时，它在内部自动完成了一系列极其复杂的分布式协同操作：
1. 加载元数据（Metadata Loading）：
- 首先，它会读取检查点目录下的元数据文件。这些文件记录了当初保存检查点时，模型是如何被切分的、使用了什么样的并行策略（TP/PP/DP）、优化器状态是如何分布的等全局信息。
2. 获取加载策略（Strategy Resolution）：
- 它会自动调用类似你之前了解的 get_default_load_sharded_strategy，获取当前环境最适合的加载策略（例如 FullyParallelLoadStrategyWrapper）。
3. 全局拓扑映射（Global Topology Mapping）：
- 策略会根据元数据和当前传入的 sharded_state_dict，计算出当前集群中每一张 GPU（Rank）具体需要从磁盘的哪些文件中，读取哪一段字节的数据。
4. 并行 IO 执行（Parallel IO Execution）：
- 所有 GPU 进程根据计算好的映射关系，同时向存储系统发起读取请求。每张卡只读取自己负责的那一小部分权重分片，直接填充到显存中对应的张量里。
5. 完整性校验（Integrity Validation）：
- 如果开启了 validate_access_integrity，它还会校验读取的数据形状、类型是否与预期完全一致，确保不会因为并行配置变更（比如从 64 卡恢复成 128 卡训练）导致权重错位。

#### 🔗 与之前代码的关联
结合你之前提供的 MegatronCheckpointManager.load_checkpoint 代码，dist_checkpointing.load 处于整个调用链的最底层：
- 用户层：MegatronCheckpointManager.load_checkpoint 负责准备路径、调用 generate_state_dict 生成接收容器。
- 中间层：load_dist_checkpointing 负责接收容器和路径，并调用底层的分布式加载接口。
- 底层核心：dist_checkpointing.load 负责真正调动所有 GPU，执行并行的磁盘读取和数据填充操作。

#### 💡 为什么它如此重要？
- 弹性恢复（Resharding）的基石：它允许你在保存和加载时使用不同的并行配置。例如，你可以用 8 张卡保存检查点，然后用 16 张卡加载并继续训练。dist_checkpointing.load 会自动处理权重的重新切分和映射，无需人工干预。
- 极致的 IO 效率：通过完全并行的加载策略，它彻底消除了传统“单卡汇总再分发”模式下的显存瓶颈和通信延迟，使得千亿、万亿参数模型的快速恢复成为可能。
- 高度的抽象与封装：对于上层开发者而言，只需要准备好 sharded_state_dict 和路径，剩下的复杂分布式协同工作全部由这个方法在内部透明地完成。

总结来说，megatron.core.dist_checkpointing.load 是大模型分布式训练中负责“读档”的终极执行者。它通过高度智能化的并行策略，确保了海量模型参数能够在成百上千张 GPU 之间被快速、准确且安全地恢复。

# save_checkpoint()

In [ ]:
    @register(dispatch_mode=Dispatch.ONE_TO_ALL)
    def save_checkpoint(self, checkpoint_path, hdfs_path=None, global_step=0, max_ckpt_to_keep=None):
        if self._is_offload_param:
            load_megatron_model_to_gpu(self.actor_module)
        self.checkpoint_mananager.save_checkpoint(
            local_path=checkpoint_path, hdfs_path=hdfs_path, global_step=global_step, max_ckpt_to_keep=max_ckpt_to_keep
        )
        torch.distributed.barrier()
        if self._is_offload_param:
            offload_megatron_model_to_cpu(self.actor_module)

## self.checkpoint_mananager.save_checkpoint

- save_checkpoint 代码是 MegatronCheckpointManager 的“存档”核心逻辑。
- 它的主要职责是：<font color='red'>将当前训练的模型权重、优化器状态、配置信息等，按照指定的策略（分布式原生格式或 HuggingFace 格式）精准地保存到本地磁盘，并支持自动清理旧存档以及上传至远程 HDFS 存储。</font>

我们可以将这段代码的执行流程拆解为以下五个核心环节来详细解读：

In [ ]:
    def save_checkpoint(self, local_path: str, hdfs_path: str = None, global_step: int = 0, max_ckpt_to_keep=None):
        
        '''
        历史存档清理与目录准备:
        
        在正式保存前，代码首先记录当前的全局步数（global_step），并根据 max_ckpt_to_keep 参数执行自动清理策略。
        如果本地保存的检查点数量超过了设定的上限，会自动删除最早的一批存档，防止磁盘空间被无限占满。
        随后，创建安全的本地保存目录，并解析出分布式检查点的具体存储路径。
        
        '''
        # record the previous global step
        self.previous_global_step = global_step

        # remove previous local_path 自动清理超出数量限制的旧检查点
        if (
            max_ckpt_to_keep
            and isinstance(max_ckpt_to_keep, int)
            and max_ckpt_to_keep > 0
            and len(self.previous_saved_paths) >= max_ckpt_to_keep
        ):
            keep_start = len(self.previous_saved_paths) - max_ckpt_to_keep + 1
            self.remove_previous_save_local_path(self.previous_saved_paths[:keep_start])
            self.previous_saved_paths = self.previous_saved_paths[keep_start:]

        local_path = local_mkdir_safe(local_path)
        dist_checkpoint_path = get_dist_checkpoint_path(local_path)

        '''
        核心权重的双模式保存（核心分支）:
        
        '''
        if self.use_dist_checkpointing:
            '''
            分支 A：纯 Megatron 分布式保存（use_dist_checkpointing）
            
            这是 Megatron 的原生高性能模式。它调用 generate_state_dict() 收集当前各 GPU 上的权重分片和优化器状态，
            然后通过 save_dist_checkpointing 将数据并行写入磁盘。这里支持 async_save（异步保存），
            可以让保存操作在后台进行，极大减少了对训练主循环的阻塞时间。
            '''
            # 生成包含模型、优化器、RNG状态的完整状态字典 Generate state dict for saving  
            state_dict = self.generate_state_dict()
            log_with_rank(f"Generated state dict for saving: {state_dict.keys()}", rank=self.rank, logger=logger)
            for vpp_rank, model in enumerate(self.model):
                if len(self.model) > 1:
                    model_i_keys = state_dict[f"model{vpp_rank}"].keys()
                    log_with_rank(f"Generated state dict for saving: {model_i_keys}", rank=self.rank, logger=logger)
                else:
                    log_with_rank(
                        f"Generated state dict for saving: {state_dict['model'].keys()}", rank=self.rank, logger=logger
                    )
            # 启动保存流程，支持异步保存（async_save） Start Async save if enabled 
            async_save_request = save_dist_checkpointing(
                sharded_state_dict=state_dict,
                ckpt_path=dist_checkpoint_path,
                async_save=self.checkpoint_config.async_save,
            )

            # Synchronize all async save requests
            if not self.checkpoint_config.async_save:
                assert async_save_request is None, "Async save request should be None when not using async save."
                torch.distributed.barrier()
        else:
            '''
            分支 B：HuggingFace 桥接保存（use_hf_checkpoint）
            
            如果处于桥接模式，代码会调用 self.bridge.save_weights，
            将 Megatron 内部的模型权重转换为标准的 HuggingFace 格式并保存，方便后续在 HF 生态中进行评估或推理。
            '''
            assert self.use_hf_checkpoint, "use_hf_checkpoint should be True when not using dist checkpointing"
            log_with_rank(f"Saving HF model checkpoint to {local_path} with bridge", rank=self.rank, logger=logger)
            hf_ckpt_path = get_hf_model_checkpoint_path(local_path)
            
            self.bridge.save_weights(self.model, hf_ckpt_path)
            log_with_rank(f"Saved bridge checkpoint to {hf_ckpt_path}", rank=self.rank, logger=logger)

        '''
        配置文件与分词器的保存（仅 Rank 0 执行）
        
        在分布式训练中，为了避免文件冲突，像模型配置（Config）、分词器（Tokenizer）这类全局共享的元数据，
        通常只由主进程（rank == 0）负责保存。
        这确保了检查点目录具备完整的 HuggingFace 格式结构，方便直接加载使用。
        '''
        if self.should_save_model:
            # Only rank 0 saves the hf config and tokenizer to huggingface path
            # No matter whether we save hf model or not
            if self.rank == 0:
                # Save tokenizer
                hf_config_tokenizer_path = get_hf_model_checkpoint_path(local_path)
                self.processing_class.save_pretrained(hf_config_tokenizer_path)
                # Save huggingface config
                self.hf_config.save_pretrained(hf_config_tokenizer_path)
                if hasattr(self.hf_config, "name_or_path") and self.hf_config.name_or_path:
                    try:
                        generation_config = GenerationConfig.from_pretrained(self.hf_config.name_or_path)
                        generation_config.save_pretrained(hf_config_tokenizer_path)
                    except Exception:
                        # if the generation config isn't available, we don't save it
                        pass
                log_with_rank(
                    f"Saved Huggingface config and tokenizer to {hf_config_tokenizer_path}",
                    rank=self.rank,
                    logger=logger,
                    log_only_rank_0=True,
                )
        '''
        额外配置与完整 HF 模型的转换保存:
        
        '''
        if self.should_save_extra:
            if self.rank == 0:
                # Save transformer config
                print(self.transformer_config)
                '''
                将 Megatron 特有的 transformer_config 转换为字典并序列化为 JSON 文件，用于记录底层的架构细节。
                '''
                transformer_config_dict = asdict(self.transformer_config)
                to_convert_types = {torch.dtype: str, AttnBackend: str}
                ignore_types = [Callable]
                pop_keys = []
                for key, value in transformer_config_dict.items():
                    if type(value) in to_convert_types:
                        transformer_config_dict[key] = to_convert_types[type(value)](value)
                    if type(value) in ignore_types:
                        pop_keys.append(key)
                    if callable(value):
                        pop_keys.append(key)
                for key in pop_keys:
                    transformer_config_dict.pop(key)
                transformer_config_path = get_transformer_config_checkpoint_path(local_path)
                '''
                ... 处理不可序列化的类型（如 torch.dtype, Callable）...
                '''
                with open(transformer_config_path, "w") as f:
                    json.dump(transformer_config_dict, f, indent=2)

        if self.should_save_hf_model:
            '''
            将分布式模型权重聚合，并转换为完整的 HF 模型格式
            
            完整 HF 模型保存：如果配置了 should_save_hf_model，代码会通过 weight_saver 将分布式切分的权重进行逻辑聚合，
            然后利用 HuggingFace 的 AutoModelForCausalLM 和 accelerate 库，生成一个完整的、可直接用于单卡推理的 HF 模型存档。
            '''
            # wait for everyone to dump to local
            state_dict = self.weight_saver(
                self.model,
                self.hf_config,
                dtype=self.param_dtype,
                is_value_model=self.is_value_model,
                tie_word_embeddings=self.share_embeddings_and_output_weights,
            )

            torch.distributed.barrier() #  等待所有进程完成分片准备
            if self.rank == 0:
                hf_model_ckpt_path = get_hf_model_checkpoint_path(local_path)
                import warnings

                from accelerate import init_empty_weights

                '''
                使用 accelerate 的空权重初始化框架加载模型结构，并注入保存好的权重
                '''
                with init_empty_weights(), warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    if "mistral7b-rm" in self.config.model.path:
                        from transformers import MistralForSequenceClassification

                        model = MistralForSequenceClassification.from_pretrained(
                            self.config.model.path
                        )  # use score head instead of lm_head
                        state_dict["score.weight"] = state_dict["score.weight"]
                    else:
                        from transformers import AutoModelForCausalLM

                        model = AutoModelForCausalLM.from_pretrained(self.config.model.path, torch_dtype="auto")
                model.save_pretrained(hf_model_ckpt_path, state_dict=state_dict)
                log_with_rank(
                    f"Saved Huggingface config and tokenizer to {hf_model_ckpt_path}",
                    rank=self.rank,
                    logger=logger,
                    log_only_rank_0=True,
                )

                if hdfs_path is not None:
                    log_with_rank(
                        f"Uploading checkpoint to {hdfs_path}", rank=self.rank, logger=logger, log_only_rank_0=True
                    )
                    from verl.utils import hdfs_io

                    hdfs_io.makedirs(hdfs_path, exist_ok=True)
                    hdfs_io.copy(src=hf_model_ckpt_path, dst=hdfs_path, dirs_exist_ok=True)
                    log_with_rank(
                        f"HDFS checkpoint uploaded to {hdfs_path}", rank=self.rank, logger=logger, log_only_rank_0=True
                    )
        '''
        异步收尾与 HDFS 远程上传:
        
        如果开启了 HDFS 路径，主进程会将本地保存好的检查点目录和配置文件同步上传到远程分布式存储，确保数据的安全归档。
        如果开启了异步保存，这个收尾函数会被注册为回调，在后台保存任务完成后自动触发；如果是同步保存，则立即执行。
        最终，将本次保存的路径记录到 previous_saved_paths 列表中，供下一次清理时使用。
        '''
        def finalize_save_fn():
            # Rank 0 uploads checkpoint to HDFS if hdfs_path is provided
            '''
            主进程负责将本地检查点上传到 HDFS 远程存储
            '''
            log_with_rank(
                f"Dist checkpointing save completed for {dist_checkpoint_path}", rank=self.rank, logger=logger
            )
            if self.rank == 0:
                if hdfs_path is not None:
                    log_with_rank(f"Uploading checkpoint to {hdfs_path}", rank=self.rank, logger=logger)
                    from verl.utils import hdfs_io

                    hdfs_io.makedirs(hdfs_path, exist_ok=True)
                    hdfs_io.copy(src=dist_checkpoint_path, dst=hdfs_path, dirs_exist_ok=True)
                    hdfs_io.copy(src=hf_config_tokenizer_path, dst=hdfs_path, dirs_exist_ok=True)

        '''
        如果是异步保存，将收尾工作（如上传HDFS）注册为回调函数
        '''
        if self.checkpoint_config.async_save:
            assert async_save_request is not None, "Async save request should not be None when using async save."
            async_save_request.add_finalize_fn(finalize_save_fn)
        else:
            '''
            同步保存则直接执行收尾工作
            '''
            finalize_save_fn()

        self.previous_saved_paths.append(local_path)


In [ ]:
from megatron.core import dist_checkpointing, mpu
from megatron.core.dist_checkpointing.serialization import (
    get_default_load_sharded_strategy,
    get_default_save_sharded_strategy,
)
from megatron.core.dist_checkpointing.strategies.fully_parallel import (
    FullyParallelLoadStrategyWrapper,
    FullyParallelSaveStrategyWrapper,
)

### save_dist_checkpointing


In [ ]:
async_save_request = save_dist_checkpointing(
                sharded_state_dict=state_dict,
                ckpt_path=dist_checkpoint_path,
                async_save=self.checkpoint_config.async_save,
            )

In [ ]:
def save_dist_checkpointing(sharded_state_dict, ckpt_path, async_save=False):
    validate_sharding_integrity = True
    # Get checkpointing strategies
    save_strategy = get_default_save_sharded_strategy("torch_dist")
    save_strategy = FullyParallelSaveStrategyWrapper(
        save_strategy, mpu.get_data_parallel_group(with_context_parallel=True)
    )

    # Save model sharded state dicts
    async_save_request = dist_checkpointing.save(
        sharded_state_dict,
        ckpt_path,
        sharded_strategy=save_strategy,
        async_sharded_save=async_save,
        validate_access_integrity=validate_sharding_integrity,
    )

    return async_save_request
